# Data Preparation for Machine Learning

This notebook prepares the engineered dataset for machine learning by selecting relevant features, checking class balance, encoding categorical variables, scaling numerical variables, and constructing a preprocessing pipeline.

The objectives of this notebook are:

- Select the final predictor variables.
- Separate the features from the target variable.
- Evaluate the balance of the target classes.
- Split the data into training and testing sets.
- Apply preprocessing techniques including encoding and scaling.
- Build a reusable preprocessing pipeline that can be used consistently during model training and deployment.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    TargetEncoder
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)

In [ ]:
from pathlib import Path


def find_project_root():
    """
    Locate the project root regardless of whether the notebook
    is launched from the project root or the notebooks folder.
    """
    current_path = Path.cwd().resolve()

    for candidate in [current_path, *current_path.parents]:
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root could not be located. "
        "Keep the data and notebooks folders inside the same project folder."
    )


PROJECT_ROOT = find_project_root()

RAW_DATA_DIRECTORY = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"
MODELS_DIRECTORY = PROJECT_ROOT / "models"

PROCESSED_DATA_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)

In [ ]:
engineered_data_path = (
    PROCESSED_DATA_DIRECTORY
    / "engineered_flight_data.csv"
)

df = pd.read_csv(
    engineered_data_path,
    parse_dates=["FL_DATE"]
)

print("Loaded from:", engineered_data_path)
print("Dataset shape:", df.shape)

df.head()

### Interpretation

The engineered dataset has been successfully loaded. This dataset contains the cleaned observations, engineered features, and the target variable required for machine learning.

In [ ]:
print("Rows :", df.shape[0])
print("Columns :", df.shape[1])

In [ ]:
df.info()

# Class Balance Analysis

Before training machine learning models, it is important to examine the distribution of the target variable.

Highly imbalanced datasets may require additional techniques such as resampling or class weighting to prevent models from favoring the majority class.

In [ ]:
class_counts = df["IS_DELAYED"].value_counts()

class_percentages = (
    df["IS_DELAYED"]
      .value_counts(normalize=True)
      .mul(100)
      .round(2)
)

balance_df = pd.DataFrame({
    "Count": class_counts,
    "Percentage (%)": class_percentages
})

balance_df.index = ["On Time", "Delayed"]

balance_df

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))

class_counts.plot(kind="bar")

plt.title("Target Variable Distribution")
plt.xlabel("Flight Status")
plt.ylabel("Number of Flights")

plt.xticks(
    [0,1],
    ["On Time","Delayed"],
    rotation=0
)

plt.show()

### Interpretation

The dataset contains approximately **83% on-time flights** and **17% delayed flights**.

Although the classes are not perfectly balanced, the minority class contains a sufficiently large number of observations. Therefore, no resampling techniques will be applied at this stage. During model training, class weighting will be considered where appropriate.

In [ ]:
df.columns.tolist()

In [ ]:
df["CANCELLED"].value_counts()

In [ ]:
df["DIVERTED"].value_counts()

# Feature Selection

The engineered dataset contains a mixture of operational, temporal, and categorical variables.

To build a pre-departure prediction model, only variables that would be available before a flight departs are retained.

The following variables are excluded:

- Identifier columns that do not contribute meaningful predictive information.
- Variables unavailable before departure.
- Constant columns with no variation.
- Redundant variables derived from other selected features.

In [ ]:
selected_features = [

    "AIRLINE",

    "ORIGIN",
    "DEST",

    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "DISTANCE",

    "YEAR",
    "MONTH",
    "DAY",
    "DAY_OF_WEEK",
    "QUARTER",

    "DEP_HOUR",
    "TIME_OF_DAY",
    "SEASON",
    "DISTANCE_CATEGORY",

    "IS_WEEKEND",
    "IS_PEAK_SEASON",
    "IS_BUSY_HOUR"

]

target = "IS_DELAYED"

In [ ]:
missing = [
    col for col in selected_features + [target]
    if col not in df.columns
]

if len(missing) == 0:
    print("All selected features are available.")
else:
    print("Missing columns:", missing)

In [ ]:
model_df = df[selected_features + [target]].copy()

print(model_df.shape)

model_df.head()

# Separate Features and Target

The selected predictor variables are separated from the target variable before splitting the dataset.

- **Features (X):** Predictor variables available before departure.
- **Target (y):** Whether the flight arrived more than 15 minutes late.

In [ ]:
X = model_df.drop(columns=target)

y = model_df[target]

print("Features shape :", X.shape)
print("Target shape   :", y.shape)

# Chronological Train-Validation-Test Split

Since this dataset contains flights from multiple years (2019–2023), a chronological split is used instead of a random split.

This approach better reflects real-world deployment, where historical data is used to predict future flight delays.

The dataset is divided as follows:

- Training Set: 2019–2021
- Validation Set: 2022
- Test Set: 2023

The validation set will be used during hyperparameter tuning, while the test set will remain completely unseen until the final model evaluation.

In [ ]:
train_df = df[df["YEAR"] <= 2021].copy()

validation_df = df[df["YEAR"] == 2022].copy()

test_df = df[df["YEAR"] == 2023].copy()

print("Training:", train_df.shape)
print("Validation:", validation_df.shape)
print("Testing:", test_df.shape)

In [ ]:
X_train = train_df[selected_features]

y_train = train_df[target]

X_validation = validation_df[selected_features]

y_validation = validation_df[target]

X_test = test_df[selected_features]

y_test = test_df[target]

In [ ]:
print("Training Features :", X_train.shape)
print("Validation Features:", X_validation.shape)
print("Testing Features  :", X_test.shape)

# Data Preprocessing

Machine learning algorithms require numerical input data.

The preprocessing pipeline performs the following transformations:

- Numerical features are standardized using StandardScaler.
- Categorical features are converted into numerical format using One-Hot Encoding.

A ColumnTransformer is used to apply the appropriate transformation to each feature type.

This preprocessing pipeline will later be combined with each machine learning algorithm to ensure identical transformations during training, evaluation, and deployment.

In [ ]:
numerical_features = [

    "CRS_DEP_TIME",
    "CRS_ARR_TIME",
    "CRS_ELAPSED_TIME",
    "DISTANCE",

    "YEAR",
    "MONTH",
    "DAY",
    "QUARTER",

    "DEP_HOUR",

    "IS_WEEKEND",
    "IS_PEAK_SEASON",
    "IS_BUSY_HOUR"

]

In [ ]:
one_hot_features = [
    "DAY_OF_WEEK",
    "TIME_OF_DAY",
    "SEASON",
    "DISTANCE_CATEGORY"
]

target_encode_features = [
    "AIRLINE",
    "ORIGIN",
    "DEST"
]

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

In [ ]:
one_hot_transformer = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [ ]:
target_transformer = Pipeline(
    steps=[
        (
            "target_encoder",
            TargetEncoder(
                target_type="binary",
                smooth="auto",
                cv=5,
                random_state=42
            )
        )
    ]
)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[

        (
            "num",
            numeric_transformer,
            numerical_features
        ),

        (
            "onehot",
            one_hot_transformer,
            one_hot_features
        ),

        (
            "target",
            target_transformer,
            target_encode_features
        )

    ],
    remainder="drop"
)

preprocessor